# k01-translate-en2vi
Generated from kaggle/jobs. Requires private input data; no outputs or metrics are fabricated.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", 'transformers==4.46.3', 'datasets>=3,<4', 'accelerate>=1,<2', 'sentence-transformers==3.3.1', 'optimum[onnxruntime]==1.23.3', 'onnx>=1.16,<2', 'onnxruntime>=1.20,<2', 'pyvi>=0.1.1', 'sentencepiece>=0.2', 'protobuf', 'pyarrow', 'pandas', 'scikit-learn', 'scipy'], check=True)


In [ ]:
"""Shared source embedded into each generated standalone Kaggle notebook."""
import os
os.environ['TOKENIZERS_PARALLELISM']='false'
import json
import random
import subprocess
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers

SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout)
print({'torch':torch.__version__,'transformers':transformers.__version__,'seed':SEED})
WORK=Path('/kaggle/working');WORK.mkdir(exist_ok=True)
INPUT=Path('/kaggle/input/ecom-intent-data')
START=time.monotonic()


def input_file(name,required=True):
    candidates=sorted(Path('/kaggle/input').rglob(name))
    if not candidates:
        if required:raise FileNotFoundError(f'Upload {name} to the private ecom-intent-data dataset')
        return None
    if len(candidates)>1:
        raise ValueError(f'Ambiguous input {name}: {candidates}. Attach only one data version.')
    return candidates[0]


def save_json(path,data):
    Path(path).write_text(json.dumps(data,ensure_ascii=False,indent=2,allow_nan=False))


def time_budget():
    if time.monotonic()-START>5.5*3600:
        raise RuntimeError('5.5 hour budget reached. Download checkpoint, attach it to the dataset, then resume.')


def normalize_text(text):
    import re,unicodedata
    return re.sub(r'\s+',' ',unicodedata.normalize('NFC',text)).strip()


def preprocess(text):return normalize_text(text).lower()


def segment_vi(text):
    from pyvi import ViTokenizer
    return ViTokenizer.tokenize(preprocess(text))


def export_onnx(source,destination,task):
    """GPU export in Kaggle; local work only consumes the exported ONNX."""
    from optimum.onnxruntime import ORTModelForFeatureExtraction,ORTModelForSequenceClassification,ORTQuantizer
    from optimum.onnxruntime.configuration import AutoQuantizationConfig
    cls=ORTModelForFeatureExtraction if task=='feature-extraction' else ORTModelForSequenceClassification
    exported=cls.from_pretrained(str(source),export=True)
    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)
    exported.save_pretrained(destination)
    quantizer=ORTQuantizer.from_pretrained(exported)
    quantizer.quantize(save_dir=destination,quantization_config=AutoQuantizationConfig.avx2(is_static=False,per_channel=False),file_suffix='int8')
    from transformers import AutoTokenizer
    AutoTokenizer.from_pretrained(str(source)).save_pretrained(destination)


In [ ]:
"""k01: translate unique protected instructions; paraphrases are delegated to k02."""
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM


def run_translation():
    if not torch.cuda.is_available():raise RuntimeError('Enable GPU and Internet on the verified Kaggle account')
    frame=pd.read_parquet(input_file('translation_input.parquet')).sort_values('text_en',key=lambda s:s.str.len())
    output=WORK/'translated_partial.parquet'
    checkpoint=input_file('translated_partial.parquet',required=False)
    done=pd.read_parquet(output if output.exists() else checkpoint).to_dict('records') if output.exists() or checkpoint else []
    frame=frame[~frame.src_id.isin({r['src_id'] for r in done})]
    name='vinai/vinai-translate-en2vi-v2';nllb=False
    try:
        tok=AutoTokenizer.from_pretrained(name,src_lang='en_XX')
        model=AutoModelForSeq2SeqLM.from_pretrained(name).half().cuda().eval()
    except (OSError,ValueError,RuntimeError) as exc:
        print('Default translator unavailable:',str(exc));name='facebook/nllb-200-distilled-600M';nllb=True
        tok=AutoTokenizer.from_pretrained(name,src_lang='eng_Latn')
        model=AutoModelForSeq2SeqLM.from_pretrained(name).half().cuda().eval()
    target='vie_Latn' if nllb else 'vi_VN'
    target_id=tok.lang_code_to_id[target] if hasattr(tok,'lang_code_to_id') else tok.convert_tokens_to_ids(target)
    if target_id is None or target_id==tok.unk_token_id:raise ValueError('Tokenizer does not recognize target language token')
    since_save=0
    try:
        for offset in range(0,len(frame),64):
            time_budget();batch=frame.iloc[offset:offset+64]
            encoded=tok(batch.text_en.tolist(),return_tensors='pt',padding=True,truncation=True,max_length=128).to('cuda')
            kwargs={'forced_bos_token_id':target_id} if nllb else {'decoder_start_token_id':target_id}
            with torch.inference_mode():out=model.generate(**encoded,**kwargs,num_beams=5,early_stopping=True,max_new_tokens=128)
            translations=tok.batch_decode(out,skip_special_tokens=True)
            for row,vi in zip(batch.to_dict('records'),translations):
                done.append({'src_id':row['src_id'],'text_en':row['text_en'],'text_vi':vi,'vi_para_1':'','vi_para_2':'','translator':name})
            since_save+=len(batch)
            if since_save>=5000:pd.DataFrame(done).to_parquet(output,index=False);since_save=0
    finally:
        if done:pd.DataFrame(done).to_parquet(output,index=False)
    pd.DataFrame(done).sort_values('src_id').to_parquet(WORK/'bitext_vi_translated.parquet',index=False)
    save_json(WORK/'translation_status.json',{'complete':True,'rows':len(done),'paraphrases':'Run k02 for train only to keep each kernel below six hours'})


run_translation()
